# 05 — Validación aritmética del pedido

**Qué hace.** Comprueba que `Σ cantidad × precio − descuento = total_dicho`, busca una explicación concreta cuando no cuadra, diagnostica si el error viene del ASR o del hablante, y puntúa el validador con las trampas plantadas en el corpus.

**Qué lee.** Las referencias de las llamadas (`data/referencias/l1…l5.txt`), `data/ground_truth/llamadas.csv` (columnas `trampa_*`, `diferencia_total` y `causa_error`), la carta y `config.yaml → validacion`.

**Qué produce.** La tabla de atrapados contra no atrapados, la comparación de cada explicación con la causa real y las pruebas del diagnóstico. Cuando haya transcripciones, también por nivel de la escalera (sección 6).

Llama a `src.extraccion_reglas`, `src.validacion` y `src.metricas`; no reimplementa nada. Las reglas están en el docstring de `src/validacion.py`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
from src.config import cargar_config
from src.extraccion_reglas import extraer, precio_carta
from src.metricas import ground_truth
from src.validacion import diagnosticar, explicar_diferencia, validar_total

cfg = cargar_config()
rutas = cfg["rutas"]
pd.set_option("display.max_colwidth", None)

## 1. Chequeo aritmético

> **Techo con transcripción perfecta.** Las llamadas se extraen con reglas a partir de las referencias (el texto del guion), que hacen de transcripción del ASR. En `03_extraccion` las 5 llamadas aciertan todos los campos, así que aquí el validador ve el pedido correcto: lo único que puede romper la suma es el error que se plantó al hablar. Con audio real se sumarán los errores del ASR.

In [2]:
llamadas = pd.read_csv(rutas["ground_truth_llamadas"]).set_index("clip")
transcripciones = {clip: " ".join((rutas["referencias_dir"] / cfg["archivos"]["patron_referencia"].format(clip=clip))
                                  .read_text(encoding="utf-8").split()) for clip in llamadas.index}
registros = {clip: extraer(texto, "pedido") for clip, texto in transcripciones.items()}

chequeo = pd.DataFrame({clip: validar_total(r) for clip, r in registros.items()}).T
chequeo

,esperado,dicho,diferencia,cuadra,motivo
l1,96,96,0,True,None
l2,90,95,5,False,None
l3,111,121,10,False,None
l4,85,85,0,True,None
l5,140,131,-9,False,None


## 2. Explicación de la diferencia

`explicar_diferencia` prueba causas de un solo paso que den exactamente la diferencia. Las de tipo **humana** son olvidar o duplicar el descuento, omitir o cobrar de más una unidad o un ítem, y cobrar un tamaño por otro. Las de tipo **ambigua** son un salto de 1, 10 o 100, que puede ser una suma mal llevada o un número mal oído. No prueba combinaciones de dos causas: con ellas casi cualquier diferencia encontraría explicación.

### 2.1 En las llamadas del corpus

In [3]:
pd.DataFrame({clip: {"diferencia": chequeo.diferencia[clip], **explicar_diferencia(r)}
              for clip, r in registros.items()}).T[["diferencia", "explicacion", "tipo", "candidatas"]]

,diferencia,explicacion,tipo,candidatas
l1,0,cuadra,None,[]
l2,5,se olvidó el descuento de promo app (5),humana,"[{'causa': 'se olvidó el descuento de promo app (5)', 'tipo': 'humana'}]"
l3,10,salto de +10: suma mal llevada o número mal oído,ambigua,"[{'causa': 'salto de +10: suma mal llevada o número mal oído', 'tipo': 'ambigua'}]"
l4,0,cuadra,None,[]
l5,-9,se omitió una unidad de gaseosa 1.5 litros (9),humana,"[{'causa': 'se omitió una unidad de gaseosa 1.5 litros (9)', 'tipo': 'humana'}]"


### 2.2 Casos inventados

Para no ajustar la función a los guiones, se prueba con 10 pedidos inventados, uno por tipo de causa. Incluyen uno sin causa simple, un salto ambiguo, uno con dos causas posibles, uno no validable y uno que cuadra. La expectativa de cada caso se fijó antes de ejecutarlo.

In [4]:
def item(producto, tamano, cantidad):
    return {"producto": producto, "tamano": tamano, "cantidad": cantidad,
            "precio_unitario": precio_carta(producto, tamano)}


casos_explicacion = [
    # id, qué simula, registro, frase que debe estar en la explicación (o explicación exacta)
    ("x01", "descuento aplicado dos veces",
     {"items": [item("pollo a la brasa", "entero", 1), item("papas fritas", "familiar", 1)],
      "promocion": "duo familiar", "descuento": 10, "total_dicho": 62}, "se aplicó dos veces el descuento"),
    ("x02", "una unidad cobrada de más",
     {"items": [item("chicha morada", "jarra", 2)], "descuento": 0, "total_dicho": 36},
     "se cobró una unidad de más de chicha morada jarra"),
    ("x03", "ítem completo omitido",
     {"items": [item("pollo a la brasa", "cuarto", 1), item("papas fritas", "personal", 3)],
      "descuento": 0, "total_dicho": 20}, "se omitió el ítem completo 3× papas fritas personal"),
    ("x04", "tamaño cobrado por otro",
     {"items": [item("pollo a la brasa", "medio", 1)], "descuento": 0, "total_dicho": 68},
     "se cobró una unidad de pollo a la brasa medio como entero"),
    ("x05", "ítem contado dos veces",
     {"items": [item("gaseosa", "personal", 3), item("torta de chocolate", "porción", 1)],
      "descuento": 0, "total_dicho": 33}, "se contó dos veces el ítem 3× gaseosa personal"),
    ("x06", "diferencia sin causa simple",
     {"items": [item("pollo a la brasa", "entero", 1)], "descuento": 0, "total_dicho": 71}, "sin explicación"),
    ("x07", "salto de una decena",
     {"items": [item("pollo a la brasa", "entero", 1), item("papas fritas", "familiar", 1)],
      "descuento": 0, "total_dicho": 92}, "salto de +10"),
    ("x08", "dos causas posibles a la vez",
     {"items": [item("chicha morada", "vaso", 2), item("ensalada clásica", "único", 1)],
      "promocion": "promo app", "descuento": 5, "total_dicho": 20}, "se olvidó el descuento"),
    ("x09", "falta un precio",
     {"items": [{"producto": "pollo a la brasa", "tamano": None, "cantidad": 1, "precio_unitario": None}],
      "descuento": 0, "total_dicho": 68}, "no validable"),
    ("x10", "total que cuadra",
     {"items": [item("anticuchos", "porción", 2)], "descuento": 0, "total_dicho": 36}, "cuadra"),
]


tabla_explicacion = pd.DataFrame([
    {"id": i, "simula": que, "diferencia": validar_total(r)["diferencia"], "esperado": esperado,
     "obtenido": (e := explicar_diferencia(r))["explicacion"], "tipo": e["tipo"],
     "n_candidatas": len(e["candidatas"]), "pasa": esperado in e["explicacion"]}
    for i, que, r, esperado in casos_explicacion])
tabla_explicacion

,id,simula,diferencia,esperado,obtenido,tipo,n_candidatas,pasa
0,x01,descuento aplicado dos veces,-10.0,se aplicó dos veces el descuento,se aplicó dos veces el descuento de duo familiar (10),humana,2,True
1,x02,una unidad cobrada de más,12.0,se cobró una unidad de más de chicha morada jarra,se cobró una unidad de más de chicha morada jarra (12),humana,1,True
2,x03,ítem completo omitido,-21.0,se omitió el ítem completo 3× papas fritas personal,se omitió el ítem completo 3× papas fritas personal (21),humana,1,True
3,x04,tamaño cobrado por otro,32.0,se cobró una unidad de pollo a la brasa medio como entero,se cobró una unidad de pollo a la brasa medio como entero (+32),humana,1,True
4,x05,ítem contado dos veces,12.0,se contó dos veces el ítem 3× gaseosa personal,se contó dos veces el ítem 3× gaseosa personal (12),humana,1,True
5,x06,diferencia sin causa simple,3.0,sin explicación,sin explicación,NaN,0,True
6,x07,salto de una decena,10.0,salto de +10,salto de +10: suma mal llevada o número mal oído,ambigua,1,True
7,x08,dos causas posibles a la vez,5.0,se olvidó el descuento,se olvidó el descuento de promo app (5),humana,2,True
8,x09,falta un precio,NaN,no validable,no validable,NaN,0,True
9,x10,total que cuadra,0.0,cuadra,cuadra,NaN,0,True


In [5]:
for i, que, r, esperado in casos_explicacion:
    obtenido = explicar_diferencia(r)["explicacion"]
    assert esperado in obtenido, f"{i} ({que}): se esperaba {esperado!r}, salió {obtenido!r}"
print(f"OK: {len(casos_explicacion)} casos de explicar_diferencia")
print("Candidatas del caso x08:", explicar_diferencia(casos_explicacion[7][2])["candidatas"])

OK: 10 casos de explicar_diferencia
Candidatas del caso x08: [{'causa': 'se olvidó el descuento de promo app (5)', 'tipo': 'humana'}, {'causa': 'se cobró una unidad de más de chicha morada vaso (5)', 'tipo': 'humana'}]


El caso x08 muestra el límite de esta función. Una diferencia de +5 se explica igual por "se olvidó el descuento de 5" que por "se cobró un vaso de chicha de más", que también cuesta 5. La función devuelve las dos candidatas y pone primero la del descuento, pero el audio no dice cuál fue. Cuantos más precios iguales tenga la carta, más a menudo pasará esto.

## 3. Atrapados contra no atrapados

Se compara el veredicto del validador con las trampas del ground truth:

- **atrapado**: el total estaba errado y el validador dice que no cuadra.
- **no atrapado**: estaba errado y el validador dice que cuadra.
- **falsa alarma**: estaba bien y el validador dice que no cuadra.
- **correcto**: estaba bien y el validador dice que cuadra.

La columna `causa_esperada` es mi lectura de `causa_error` del ground truth, expresada como el fragmento que debería aparecer en la explicación.

In [6]:
causa_esperada = {  # causa_error del ground truth -> fragmento esperado en la explicación
    "Hablante: el vendedor olvidó el descuento": "se olvidó el descuento",
    "Hablante: el vendedor sumó mal": "suma mal llevada",
    "Hablante: el vendedor omitió una gaseosa": "se omitió una unidad de gaseosa",
}


def veredicto(errado, cuadra):
    if errado:
        return "atrapado" if not cuadra else "no atrapado"
    return "falsa alarma" if not cuadra else "correcto"


filas = []
for clip, r in registros.items():
    gt = llamadas.loc[clip]
    v, e = validar_total(r), explicar_diferencia(r)
    esperada = causa_esperada.get(gt["causa_error"]) if isinstance(gt["causa_error"], str) else None
    filas.append({
        "clip": clip, "total_errado": gt["trampa_total_errado"], "cuadra": v["cuadra"],
        "veredicto": veredicto(gt["trampa_total_errado"], v["cuadra"]),
        "diferencia_gt": gt["diferencia_total"], "diferencia_detectada": v["diferencia"],
        "causa_real": gt["causa_error"], "explicacion": e["explicacion"], "tipo": e["tipo"],
        "explicacion_coincide": None if esperada is None else esperada in e["explicacion"],
    })
atrapados = pd.DataFrame(filas).set_index("clip")
atrapados

,total_errado,cuadra,veredicto,diferencia_gt,diferencia_detectada,causa_real,explicacion,tipo,explicacion_coincide
clip,,,,,,,,,
l1,False,True,correcto,0,0,NaN,cuadra,NaN,None
l2,True,False,atrapado,5,5,Hablante: el vendedor olvidó el descuento,se olvidó el descuento de promo app (5),humana,True
l3,True,False,atrapado,10,10,Hablante: el vendedor sumó mal,salto de +10: suma mal llevada o número mal oído,ambigua,True
l4,False,True,correcto,0,0,NaN,cuadra,NaN,None
l5,True,False,atrapado,-9,-9,Hablante: el vendedor omitió una gaseosa,se omitió una unidad de gaseosa 1.5 litros (9),humana,True


In [7]:
resumen = atrapados.veredicto.value_counts().reindex(["atrapado", "no atrapado", "falsa alarma", "correcto"], fill_value=0)
print(resumen.to_string())
errados = atrapados[atrapados.total_errado]
print(f"\nDiferencia detectada igual a la del ground truth: {(errados.diferencia_gt == errados.diferencia_detectada).sum()} de {len(errados)}")
print(f"Explicación que coincide con la causa real: {errados.explicacion_coincide.sum()} de {len(errados)}"
      f" (de tipo humana: {(errados.tipo == 'humana').sum()})")

veredicto
atrapado        3
no atrapado     0
falsa alarma    0
correcto        2

Diferencia detectada igual a la del ground truth: 3 de 3
Explicación que coincide con la causa real: 3 de 3 (de tipo humana: 2)


## 4. Diagnóstico: ¿ASR o hablante?

`diagnosticar(registro, palabras)` combina tres señales:

- **a) probabilidad**: la mínima de las palabras numéricas del total y de los ítems. Por debajo de `umbral_probabilidad_palabra` (0.6, provisional) es señal de ASR.
- **b) repetición**: el total se dice dos o más veces igual. Es señal de hablante.
- **c) explicación humana**: es señal de hablante.

Si solo hay señal de ASR, el diagnóstico es `asr`. Si hay señales de los dos lados, `indeterminado`. Si solo hay de hablante, `hablante`. Sin ninguna, `indeterminado`.

### 4.1 En las llamadas del corpus, sin probabilidades

Las referencias no tienen probabilidades por palabra, así que la señal a) no existe: con estas transcripciones el diagnóstico solo puede ser `hablante` o `indeterminado`. Las tres causas reales del corpus son del hablante.

In [8]:
def palabras_sin_probabilidad(texto):
    return [{"word": w, "probability": None} for w in texto.split()]


diagnostico_corpus = pd.DataFrame([
    {"clip": clip, "causa_real": llamadas.loc[clip, "causa_error"],
     **{k: d[k] for k in ("diagnostico", "razon")},
     "menciones_total": d["senales"].get("menciones_total")}
    for clip, r in registros.items() if not validar_total(r)["cuadra"]
    for d in [diagnosticar(r, palabras_sin_probabilidad(transcripciones[clip]))]
]).set_index("clip")
diagnostico_corpus

,causa_real,diagnostico,razon,menciones_total
clip,,,,
l2,Hablante: el vendedor olvidó el descuento,hablante,sin probabilidades por palabra: solo se usan la repetición y la explicación; el total 95 se dice 2 veces: es poco probable que el ASR se equivoque igual dos veces; explicación: se olvidó el descuento de promo app (5) (humana),2
l3,Hablante: el vendedor sumó mal,indeterminado,sin probabilidades por palabra: solo se usan la repetición y la explicación; el total 121 se dice 1 vez; explicación: salto de +10: suma mal llevada o número mal oído (ambigua),1
l5,Hablante: el vendedor omitió una gaseosa,hablante,sin probabilidades por palabra: solo se usan la repetición y la explicación; el total 131 se dice 2 veces: es poco probable que el ASR se equivoque igual dos veces; explicación: se omitió una unidad de gaseosa 1.5 litros (9) (humana),2


In [9]:
aciertos_diag = (diagnostico_corpus.diagnostico == "hablante").sum()
print(f"Diagnóstico correcto ('hablante'): {aciertos_diag} de {len(diagnostico_corpus)}")

Diagnóstico correcto ('hablante'): 2 de 3


### 4.2 Casos inventados, con probabilidades

Nueve transcripciones inventadas, extraídas con las mismas reglas, cubren las tres salidas y el caso que cuadra. Las palabras se construyen como las da faster-whisper: probabilidad 0.95 por defecto, y en algunos casos una palabra concreta con probabilidad baja. La expectativa de cada caso se fijó antes de ejecutarlo.

In [10]:
def palabras(texto, probabilidad=0.95, bajas=None):
    """Palabras como las da faster-whisper; `bajas` fija la probabilidad de palabras concretas."""
    bajas = bajas or {}
    return [{"word": w, "probability": None if probabilidad is None else
             bajas.get(w.strip(" ¿?¡!.,;:").lower(), probabilidad)} for w in texto.split()]


pedido = "Quiero un pollo entero y una papa familiar. Le confirmo: un pollo entero y una papa familiar."
casos_diagnostico = [
    # id, qué simula, transcripción, probabilidades, esperado
    ("d01", "total mal oído: probabilidad baja, sin repetición, salto ambiguo",
     pedido + " El total es noventa y dos soles. Perfecto.", dict(bajas={"noventa": 0.35}), "asr"),
    ("d02", "total mal oído y sin explicación",
     pedido + " El total es ochenta y cinco soles.", dict(bajas={"cinco": 0.41}), "asr"),
    ("d03", "descuento olvidado, todo con probabilidad alta",
     "Tengo el código LEÑA cinco. " + pedido + " El total es ochenta y dos soles.", {}, "hablante"),
    ("d04", "probabilidad alta pero ninguna explicación ni repetición",
     pedido + " El total es ochenta y cinco soles.", {}, "indeterminado"),
    ("d05", "conflicto: probabilidad baja, pero el cliente repite el total",
     pedido + " El total es noventa y dos soles. Noventa y dos, ok.", dict(bajas={"noventa": 0.30}), "indeterminado"),
    ("d06", "cantidad mal oída ('tres' por 'dos'): la diferencia parece una unidad omitida",
     "Le confirmo: un pollo entero y tres jarras de chicha morada. El total es noventa y dos soles.",
     dict(bajas={"tres": 0.38}), "indeterminado"),
    ("d07", "sin probabilidades, el cliente repite el total",
     pedido + " El total es ochenta y cinco soles. Ochenta y cinco, listo.", dict(probabilidad=None), "hablante"),
    ("d08", "sin probabilidades y sin otra señal",
     pedido + " El total es ochenta y cinco soles.", dict(probabilidad=None), "indeterminado"),
    ("d09", "total que cuadra",
     pedido + " El total es ochenta y dos soles.", {}, None),
]


tabla_diagnostico = pd.DataFrame([
    {"id": i, "simula": que, "esperado": esperado, "obtenido": (d := diagnosticar(
        extraer(texto, "pedido"), palabras(texto, **probabilidades)))["diagnostico"],
     "pasa": d["diagnostico"] == esperado, "razon": d["razon"]}
    for i, que, texto, probabilidades, esperado in casos_diagnostico])
tabla_diagnostico

,id,simula,esperado,obtenido,pasa,razon
0,d01,"total mal oído: probabilidad baja, sin repetición, salto ambiguo",asr,asr,True,palabra numérica con probabilidad baja: 'noventa' = 0.35 < 0.6; el total 92 se dice 1 vez; explicación: salto de +10: suma mal llevada o número mal oído (ambigua)
1,d02,total mal oído y sin explicación,asr,asr,True,palabra numérica con probabilidad baja: 'cinco' = 0.41 < 0.6; el total 85 se dice 1 vez; explicación: sin explicación
2,d03,"descuento olvidado, todo con probabilidad alta",hablante,hablante,True,todas las palabras numéricas con probabilidad >= 0.6 (mínima 0.95); no descarta al ASR por sí sola; el total 82 se dice 1 vez; explicación: se olvidó el descuento de promo app (5) (humana)
3,d04,probabilidad alta pero ninguna explicación ni repetición,indeterminado,indeterminado,True,todas las palabras numéricas con probabilidad >= 0.6 (mínima 0.95); no descarta al ASR por sí sola; el total 85 se dice 1 vez; explicación: sin explicación
4,d05,"conflicto: probabilidad baja, pero el cliente repite el total",indeterminado,indeterminado,True,conflicto entre señales; palabra numérica con probabilidad baja: 'noventa' = 0.30 < 0.6; el total 92 se dice 2 veces: es poco probable que el ASR se equivoque igual dos veces; explicación: salto de +10: suma mal llevada o número mal oído (ambigua)
5,d06,cantidad mal oída ('tres' por 'dos'): la diferencia parece una unidad omitida,indeterminado,indeterminado,True,conflicto entre señales; palabra numérica con probabilidad baja: 'tres' = 0.38 < 0.6; el total 92 se dice 1 vez; explicación: se omitió una unidad de chicha morada jarra (12) (humana)
6,d07,"sin probabilidades, el cliente repite el total",hablante,hablante,True,sin probabilidades por palabra: solo se usan la repetición y la explicación; el total 85 se dice 2 veces: es poco probable que el ASR se equivoque igual dos veces; explicación: sin explicación
7,d08,sin probabilidades y sin otra señal,indeterminado,indeterminado,True,sin probabilidades por palabra: solo se usan la repetición y la explicación; el total 85 se dice 1 vez; explicación: sin explicación
8,d09,total que cuadra,NaN,NaN,True,el total cuadra: no hay nada que diagnosticar


In [11]:
for i, que, texto, probabilidades, esperado in casos_diagnostico:
    obtenido = diagnosticar(extraer(texto, "pedido"), palabras(texto, **probabilidades))["diagnostico"]
    assert obtenido == esperado, f"{i} ({que}): se esperaba {esperado!r}, salió {obtenido!r}"
salidas = set(tabla_diagnostico.obtenido.dropna())
assert salidas == {"asr", "hablante", "indeterminado"}, salidas
print(f"OK: {len(casos_diagnostico)} casos de diagnosticar; salidas cubiertas: {sorted(salidas)}")

OK: 9 casos de diagnosticar; salidas cubiertas: ['asr', 'hablante', 'indeterminado']


## 5. Qué atrapa el validador y qué no

**Qué atrapa (corpus, transcripción perfecta)**
- Los 3 totales errados plantados (L2, L3 y L5) quedan atrapados, sin falsas alarmas en L1 ni en L4, y la diferencia detectada es exactamente la del ground truth.
- Para L2 ("olvidó el descuento", +5) y L5 ("omitió una gaseosa", −9), la explicación coincide con la causa real, es de tipo humana y el diagnóstico es `hablante`. En las dos llamadas, además, el cliente repite el total.

**Qué no atrapa o no explica**
- **L3, "sumó mal" (+10).** El validador lo atrapa, pero la única explicación es un salto de +10, que es ambiguo: un vendedor que suma mal y un ASR que oye "ciento veintiuno" en lugar de "ciento once" producen la misma diferencia. Como el total se dice una sola vez, el diagnóstico queda `indeterminado`, aunque la causa real es del hablante. Es la respuesta honesta con la información disponible.
- **Una diferencia de 0 no prueba que el pedido esté bien.** Si el ASR oye mal un ítem y también el total, de forma que la suma siga cuadrando, el validador no lo ve. Solo comprueba la coherencia interna del registro, no que coincida con lo que se dijo.
- **Explicación ambigua entre causas** (caso x08). Si dos causas dan la misma diferencia, la función no puede elegir entre ellas a partir del audio.
- **La explicación no separa al ASR del hablante** (caso d06). Si el ASR oye "tres jarras" en lugar de "dos", la diferencia se parece a "se omitió una jarra". Solo la probabilidad baja de "tres" permite sospechar, y entonces el diagnóstico queda `indeterminado` por conflicto, no `asr`.
- **Sin probabilidades no hay diagnóstico `asr`.** Con las referencias, la regla nunca puede concluir que el ASR falló. El diagnóstico de verdad necesita las transcripciones de faster-whisper, con `word_timestamps=True`.

El umbral de 0.6 es provisional. Hay que calibrarlo con las probabilidades reales de las palabras bien y mal transcritas (sección 6).

## 6. Diagnóstico con audio real y escalera de degradación

Validación y diagnóstico sobre las transcripciones de faster-whisper (con probabilidad por palabra) en cada nivel L0–L3: atrapados contra no atrapados por nivel, exactitud del diagnóstico contra la causa real (los errores del ASR aparecerán al degradar el audio) y calibración de `umbral_probabilidad_palabra`.

*Pendiente: se llena cuando existan las transcripciones de `02_transcripcion_gpu` y `04_degradacion`.*